# Decorator Design Pattern 

explained using the classic Coffee Shop example.

#### The Concept

The Decorator Pattern allows you to add features to an object dynamically by **"wrapping"** it in another object. Instead of creating a giant class hierarchy like CoffeeWithMilk, CoffeeWithSugar, CoffeeWithMilkAndSugar, you start with a plain Coffee and wrap it in layers.

**Analogy**: Putting on clothes. You start with your body (Component). You put on a shirt (Decorator 1). Then a jacket (Decorator 2). You are still "You", but with added layers.

## The Classic OOP Way (Java-Style)

In the Java/C++ style, we use an Abstract Base Class for the component, and a specific "Decorator" class that mimics the component while holding a reference to it.

#### THE COMPONENT INTERFACE

In [1]:
from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def get_cost(self) -> float:
        pass

    @abstractmethod
    def get_description(self) -> str:
        pass

#### CONCRETE COMPONENT (The Base Object)

In [2]:
class SimpleCoffee(Coffee):
    def get_cost(self) -> float:
        return 5.0

    def get_description(self) -> str:
        return "Simple Coffee"

#### THE BASE DECORATOR

In [3]:
class CoffeeDecorator(Coffee):
    """
    The 'Middleman'. It looks like a Coffee, but it holds a reference
    to another Coffee object inside it.
    """
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

    def get_cost(self) -> float:
        return self._coffee.get_cost()

    def get_description(self) -> str:
        return self._coffee.get_description()

#### CONCRETE DECORATORS (The Layers)

In [4]:
class Milk(CoffeeDecorator):
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 1.5

    def get_description(self) -> str:
        return self._coffee.get_description() + ", Milk"

class Sugar(CoffeeDecorator):
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.5

    def get_description(self) -> str:
        return self._coffee.get_description() + ", Sugar"

#### CLIENT CODE

In [6]:
def main():
    # 1. Start with plain coffee
    my_coffee = SimpleCoffee()
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

    # 2. Decorate with Milk
    my_coffee = Milk(my_coffee)
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

    # 3. Decorate with Sugar
    my_coffee = Sugar(my_coffee)
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

    # 4. Decorate with Sugar
    my_coffee = Sugar(my_coffee)
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

if __name__ == "__main__":
    main()

Simple Coffee : $5.0
Simple Coffee, Milk : $6.5
Simple Coffee, Milk, Sugar : $7.0
Simple Coffee, Milk, Sugar, Sugar : $7.5


## The Pythonic Way

In Python, we can leverage `__getattr__` (Dynamic Delegation). Strictly inheriting from a `CoffeeDecorator` class is often unnecessary boilerplate. We can make a wrapper that automatically forwards any unknown method call to the wrapped object. This is much more flexible.

#### PROTOCOL (Optional Type Safety)

In [8]:
from typing import Protocol

class Beverage(Protocol):
    def cost(self) -> float: ...
    def desc(self) -> str: ...

#### CONCRETE COMPONENT

In [9]:
class Espresso:
    def cost(self) -> float: return 5.0
    def desc(self) -> str: return "Espresso"

#### PYTHONIC DECORATORS (Dynamic Wrappers)

In [12]:
from typing import Any

class Milk:
    def __init__(self, wrapped: Any):
        self._wrapped = wrapped

    def cost(self) -> float:
        return self._wrapped.cost() + 1.5

    def desc(self) -> str:
        return self._wrapped.desc() + ", Milk"
    
    # MAGIC METHOD: Dynamic Delegation
    # If the user calls a method that Milk doesn't have,
    # pass it down to the wrapped object automatically.
    def __getattr__(self, name):
        return getattr(self._wrapped, name)


class Vanilla:
    def __init__(self, wrapped: Any):
        self._wrapped = wrapped

    def cost(self) -> float:
        return self._wrapped.cost() + 2.0

    def desc(self) -> str:
        return self._wrapped.desc() + ", Vanilla"

    def __getattr__(self, name):
        return getattr(self._wrapped, name)

#### CLIENT CODE

In [13]:
def main():
    # 1. Nesting Objects directly
    # Usage: Vanilla( Milk( Espresso() ) )
    
    my_drink = Vanilla(Milk(Espresso()))

    print(f"Order: {my_drink.desc()}")
    print(f"Total: ${my_drink.cost()}")

    # 5. Accessing methods that weren't explicitly overridden?
    # Because of __getattr__, if Espresso had a method 'brew()',
    # my_drink.brew() would still work!

if __name__ == "__main__":
    main()

Order: Espresso, Milk, Vanilla
Total: $8.5


| Key Differences | Feature      | Classic OOP                                        | Pythonic                                              |
|-----------------|--------------|----------------------------------------------------|--------------------------------------------------------|
| Structure       | Approach     | Rigid inheritance (extends `CoffeeDecorator`).     | Loose composition (Duck Typing).                       |
| Boilerplate     | Code Effort  | High — must implement every method from Interface. | Low — only implement methods you want to change.       |
| Delegation      | Behavior     | Manual (`return super.method()`).                  | Automatic (`__getattr__` delegates everything else).    |


Important Note on Python's `@decorator`

Python has a built-in syntax `@decorator_name` used for functions and classes. While related (they both wrap things), `the GoF Decorator Pattern` described above is about runtime object composition, whereas Python's @decorator syntax is usually for `definition-time modification`.

# Decorator Design Pattern 

explained using a complex, real-world scenario: **Web Server Middleware (Request Processing Pipeline)**.

#### The Scenario: API Request Handling

In a web framework (like Spring Boot, Django, or Express), a request comes in and needs to go through several "layers" before it reaches the actual business logic:
- **Authentication**: Is the user logged in?
- **Logging**: Record the request time.
- **Compression**: Gzip the response before sending it back.
- **Business Logic**: Fetch data from DB.

**The Problem**: Using inheritance (`AuthenticatedHandler`, `LoggedAuthenticatedHandler`, `GzipLoggedAuthenticatedHandler`) creates a class explosion. **The Solution**: Use Decorators to wrap the handler dynamically like an onion. Each layer does its job and calls the next layer.

## The Classic OOP Way (Java-Style)

In strict OOP, we use an Interface for the `Handler`. An abstract `BaseDecorator` implements this interface and holds a reference to the wrapped object.

#### THE INTERFACE

In [14]:
from abc import ABC, abstractmethod

class IRequestHandler(ABC):
    @abstractmethod
    def handle_request(self, request: str) -> str:
        pass

#### THE CORE COMPONENT (Business Logic)

In [15]:
class BasicApiHandler(IRequestHandler):
    def handle_request(self, request: str) -> str:
        print("   ⚙️ [Core] Fetching data from Database...")
        return f"{{'data': 'Results for {request}'}}"

#### THE BASE DECORATOR

In [16]:
class BaseDecorator(IRequestHandler):
    def __init__(self, wrapped: IRequestHandler):
        self._wrapped = wrapped

    def handle_request(self, request: str) -> str:
        # Default behavior: Just pass it through
        return self._wrapped.handle_request(request)

#### CONCRETE DECORATORS (Middleware Layers)

In [20]:
import time

class AuthDecorator(BaseDecorator):
    def handle_request(self, request: str) -> str:
        # 1. Pre-processing (Security Check)
        if "ADMIN" not in request:
            return "{'error': '403 Forbidden'}"
        
        print("   🛡️ [Auth] User authorized.")
        # 2. Pass to next layer
        return self._wrapped.handle_request(request)

class LoggingDecorator(BaseDecorator):
    def handle_request(self, request: str) -> str:
        print(f"   📝 [Log] Request started for: {request}")
        start_time = time.time()
        
        # Pass to next layer
        response = self._wrapped.handle_request(request)
        
        # 3. Post-processing
        duration = (time.time() - start_time) * 1000
        print(f"   📝 [Log] Request finished in {duration:.2f}ms")
        return response

class GzipDecorator(BaseDecorator):
    def handle_request(self, request: str) -> str:
        # Get original response
        response = self._wrapped.handle_request(request)
        
        # Compress it
        print("   📦 [Gzip] Compressing response...")
        return f"<GZIP>{response}</GZIP>"

#### CLIENT CODE

In [21]:
def main():
    print("--- Java-Style Middleware Stack ---")

    # 1. Create the Core
    api = BasicApiHandler()

    # 2. Wrap it in layers (The "Onion")
    # Order: Gzip -> Logging -> Auth -> Core
    # Execution flows: Outer -> Inner -> Outer
    
    secure_api = AuthDecorator(api)
    logged_api = LoggingDecorator(secure_api)
    final_stack = GzipDecorator(logged_api)

    # 3. Execute
    # Case A: Success
    print("\nRequest 1 (Admin):")
    result = final_stack.handle_request("GET_DASHBOARD (ADMIN)")
    print(f"Final Response: {result}")

    # Case B: Failure (Blocked by Auth layer, Core never reached)
    print("\nRequest 2 (Hacker):")
    result = final_stack.handle_request("GET_PASSWORDS (HACKER)")
    print(f"Final Response: {result}")

if __name__ == "__main__":
    main()

--- Java-Style Middleware Stack ---

Request 1 (Admin):
   📝 [Log] Request started for: GET_DASHBOARD (ADMIN)
   🛡️ [Auth] User authorized.
   ⚙️ [Core] Fetching data from Database...
   📝 [Log] Request finished in 0.08ms
   📦 [Gzip] Compressing response...
Final Response: <GZIP>{'data': 'Results for GET_DASHBOARD (ADMIN)'}</GZIP>

Request 2 (Hacker):
   📝 [Log] Request started for: GET_PASSWORDS (HACKER)
   📝 [Log] Request finished in 0.00ms
   📦 [Gzip] Compressing response...
Final Response: <GZIP>{'error': '403 Forbidden'}</GZIP>


## The Pythonic Way (Function Decorators)

Python has the Decorator Pattern built into the language syntax (`@decorator`). We don't need classes to wrap functionality; we can wrap functions with other functions (Closures). This is how frameworks like Flask and FastAPI work (`@app.route`, `@login_required`).

This approach reduces 50 lines of boilerplate code into just a few function definitions.

#### THE DECORATORS (Middleware Functions)

In [22]:
import time
import functools

def auth_middleware(func):
    """Checks if 'ADMIN' is in the request args."""
    @functools.wraps(func) # Preserves metadata (name, docstring)
    def wrapper(request, *args, **kwargs):
        if "ADMIN" not in request:
            return "{'error': '403 Forbidden'}"
        
        print("   🛡️ [Auth] User authorized.")
        return func(request, *args, **kwargs)
    return wrapper

def logging_middleware(func):
    """Logs timing."""
    @functools.wraps(func)
    def wrapper(request, *args, **kwargs):
        print(f"   📝 [Log] Request started: {request}")
        start_time = time.time()
        
        response = func(request, *args, **kwargs)
        
        duration = (time.time() - start_time) * 1000
        print(f"   📝 [Log] Finished in {duration:.2f}ms")
        return response
    return wrapper

def gzip_middleware(func):
    """Compresses result."""
    @functools.wraps(func)
    def wrapper(request, *args, **kwargs):
        response = func(request, *args, **kwargs)
        print("   📦 [Gzip] Compressing...")
        return f"<GZIP>{response}</GZIP>"
    return wrapper

#### THE CORE COMPONENT (Applied via Syntax)

In [23]:
# The pattern is applied vertically. 
# It reads: Gzip wraps Logging, which wraps Auth, which wraps the function.
@gzip_middleware
@logging_middleware
@auth_middleware
def get_dashboard(request):
    print("   ⚙️ [Core] Fetching DB data...")
    return f"{{'data': 'Dashboard for {request}'}}"

#### CLIENT CODE

In [24]:
def main():
    print("--- Pythonic Decorators ---")

    # 1. Valid Request
    print("\nRequest 1 (Admin):")
    # We just call the function normally. The decorators activate automatically.
    result = get_dashboard("USER:ADMIN") 
    print(f"Final: {result}")

    # 2. Invalid Request
    print("\nRequest 2 (Guest):")
    result = get_dashboard("USER:GUEST")
    print(f"Final: {result}")

if __name__ == "__main__":
    main()

--- Pythonic Decorators ---

Request 1 (Admin):
   📝 [Log] Request started: USER:ADMIN
   🛡️ [Auth] User authorized.
   ⚙️ [Core] Fetching DB data...
   📝 [Log] Finished in 0.02ms
   📦 [Gzip] Compressing...
Final: <GZIP>{'data': 'Dashboard for USER:ADMIN'}</GZIP>

Request 2 (Guest):
   📝 [Log] Request started: USER:GUEST
   📝 [Log] Finished in 0.00ms
   📦 [Gzip] Compressing...
Final: <GZIP>{'error': '403 Forbidden'}</GZIP>


#### Key Differences

| Feature          | Classic OOP                                                     | Pythonic (`@` decorators)                                       |
|------------------|-----------------------------------------------------------------|------------------------------------------------------------------|
| **Structure**    | Classes wrapping classes.                                       | Functions wrapping functions.                                   |
| **Application** | Dynamic / runtime composition (e.g., `new A(new B(new C()))`).  | Definition-time composition (`@A`, `@B` above a function).      |
| **Boilerplate**  | High — requires interfaces, base classes, and `__init__`.       | Zero — just define a function that accepts `func`.              |
| **Flexibility**  | Easy to construct different stacks at runtime.                  | Slightly static, but runtime wrapping is still possible.        |


#### When to use which?

- **Java Way**: Use this in Python only if you need to build the stack dynamically at runtime (e.g., toggling the "Gzip" layer on/off based on a user configuration checkbox).
- **Pythonic Way**: Use the `@decorator` syntax for 99% of cases (Logging, Auth, Caching, Validation). It is the standard way to apply cross-cutting concerns in Python.